## Import data and get HPV

In [1]:
!apt-get install -y bedtools
!apt-get install -y samtools
!pip install biopython

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  bedtools
0 upgraded, 1 newly installed, 0 to remove and 35 not upgraded.
Need to get 563 kB of archives.
After this operation, 1,548 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 bedtools amd64 2.30.0+dfsg-2ubuntu0.1 [563 kB]
Fetched 563 kB in 1s (591 kB/s)
Selecting previously unselected package bedtools.
(Reading database ... 126281 files and directories currently installed.)
Preparing to unpack .../bedtools_2.30.0+dfsg-2ubuntu0.1_amd64.deb ...
Unpacking bedtools (2.30.0+dfsg-2ubuntu0.1) ...
Setting up bedtools (2.30.0+dfsg-2ubuntu0.1) ...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3 libhtscodecs2
Suggested packages:
  cwltool
The following NEW packages will be inst

In [2]:
import pandas as pd
from Bio import SeqIO

In [3]:
df = pd.read_csv('VIS_table_.csv', encoding = 'latin-1')

In [4]:
df['Virus type'].value_counts()

,count
Virus type,
HIV,88935
HPV,70165
HBV,62761
HTLV-1,51806
EBV,1809
BKPyV,1523
MCV,190
HHV7,187
AAV2,33


In [5]:
df = df[df['Virus type'] == 'HPV']
df.count()

,0
VIS_ID,70165
Human chromosome,70165
hg38_start,70165
hg38_end,70165
Virus type,70165


In [6]:
# Replace end values with start values
df.loc[:, 'hg38_end'] = df['hg38_start']

In [7]:
# Drop rows with null start values.
df = df.dropna(subset=['hg38_start'])

In [8]:
# Convert to numeric with NaNs for errors, then dropna
df.loc[:, 'hg38_start'] = pd.to_numeric(df['hg38_start'], errors='coerce')
df.loc[:, 'hg38_end'] = pd.to_numeric(df['hg38_end'], errors='coerce')

df = df.dropna(subset=['hg38_start'])

# Integers
df.loc[:, 'hg38_start'] = df['hg38_start'].astype(int)
df.loc[:, 'hg38_end'] = df['hg38_end'].astype(int)

In [9]:
# Filter for start values only greater than 0
df = df[df['hg38_start'] >= 0]

In [10]:
# Create bed structured file from df
df_bed = pd.DataFrame({
    'chrom': df['Human chromosome'],
    'start': df['hg38_start'],
    'end': df['hg38_end'],
    'name': df['VIS_ID'],
    'virus': df['Virus type']
})

## Flank and save as .bed

In [11]:
df_bed['start'] = df_bed['start'] - 500
df_bed['end'] = df_bed['start'] + 1000
df_bed = df_bed[df_bed['start'] >= 0]

df_bed['end'] = df_bed.apply(lambda row: max(row['start'], row['end']), axis=1)

df_bed.to_csv('VIS_flanked.bed', sep='\t', header=False, index=False)

In [12]:
!head VIS_flanked.bed

chr9	99841705	99842705	TVIS20012973	HPV
chr3	169299140	169300140	TVIS20012974	HPV
chr7	5276228	5277228	TVIS20012975	HPV
chr7	5276189	5277189	TVIS20012976	HPV
chr3	195825139	195826139	TVIS20012977	HPV
chr1	38001640	38002640	TVIS20012978	HPV
chr1	38001638	38002638	TVIS20012979	HPV
chr1	38001636	38002636	TVIS20012980	HPV
chr1	38001635	38002635	TVIS20012981	HPV
chr1	38001591	38002591	TVIS20012982	HPV


## getfasta

In [14]:
!wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/GRCh38.primary_assembly.genome.fa.gz

--2025-07-18 00:39:38--  https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/GRCh38.primary_assembly.genome.fa.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 845635028 (806M) [application/x-gzip]
Saving to: ‘GRCh38.primary_assembly.genome.fa.gz’

GRCh38.primary_asse 100%[===================>] 806.46M  4.03MB/s    in 3m 51s  

2025-07-18 00:43:30 (3.49 MB/s) - ‘GRCh38.primary_assembly.genome.fa.gz’ saved [845635028/845635028]



In [18]:
# Step 2: Unzip
!gunzip GRCh38.primary_assembly.genome.fa.gz

# Step 3: Index the FASTA
!samtools faidx GRCh38.primary_assembly.genome.fa

# Step 4: Extract sequences using bedtools
!bedtools getfasta -fi GRCh38.primary_assembly.genome.fa -bed VIS_flanked.bed -fo HPV.fa

gzip: GRCh37.p13.genome.fa.gz: No such file or directory
^C
index file GRCh37.p13.genome.fa.fai not found, generating...
Feature (chr20:63208465-63209465) beyond the length of chr20 size (63025520 bp).  Skipping.
Feature (chr20:63208464-63209464) beyond the length of chr20 size (63025520 bp).  Skipping.
Feature (chr5:181072665-181073665) beyond the length of chr5 size (180915260 bp).  Skipping.
Feature (chr17:81838265-81839265) beyond the length of chr17 size (81195210 bp).  Skipping.
WARNING. chromosome (4) was not found in the FASTA file. Skipping.
WARNING. chromosome (11) was not found in the FASTA file. Skipping.
WARNING. chromosome (6) was not found in the FASTA file. Skipping.
WARNING. chromosome (3) was not found in the FASTA file. Skipping.
WARNING. chromosome (3) was not found in the FASTA file. Skipping.
WARNING. chromosome (15) was not found in the FASTA file. Skipping.
WARNING. chromosome (11) was not found in the FASTA file. Skipping.
WARNING. chromosome (13) was not found

## Create VIS_flanked.fa

In [16]:
!ls -lh HPV.fa
!wc -l HPV.fa

-rw-r--r-- 1 root root 68M Jul 18 00:44 HPV.fa
138158 HPV.fa


In [ ]:
!ls -lh HPV.fa

-rw-r--r-- 1 root root 68M Jul 17 14:06 HPV.fa


In [ ]:
# prompt: count lines in HBV.fa

!wc -l HPV.fa

138158 HPV.fa


In [ ]:
!head HPV.fa

>chr9:99841705-99842705
GGAGTTCAAGACCAGTGTAGGCAATATGGGAGTTCAAGACCTGCCTAGGCAATATAGGAGTTCAAGACCAGCCTAGGCAATATAGAGACACTGCCTCTACAAAAAATAGAAAAAATTAGATGGGCGTGGTGGTTCCTGCCTGTGGTTCCAGCTACTCAAGAGGCTGAGGTGGGAGGATCGCCTCAGCTGGGGAAGTCGAGGCTGTAGTGAGCCCTGATCACACCATTGCACTCCAGCCTGAGGAACAGAGCGAGACCCTGTCTCAAAAAAAAATTTTTTTTGAGGAAGGTATCCCCATTTTACAGAGTAGAAAGCTGAGGTTCATAAAAGCAAAATGATCCTACATTATTTCCCATATTAGGGGCCAGAACAAAATTGAGATTAAAGTAGCTGGACTCTCTGATTTTTTTTTTTTTTTTTTTAGTTATTTAATACTGCTTCTTGGGAAAACACAAAGAAAAAAAGACCCAATTAAATAAGCGTCTCATTTCCTCTCACCCAGGTAGCTCCACAGTAAGCACCTAGTTGCTTATTAGCTGTTTCTTCTCCAACCAGGTTAGGATATTACAGCAAACCTAAGCACTCTTCTGACTATTCTTTCATTGAAAAAATATTCAACAGTTATTTATTGAGGTGTAATTATGTGCCAGTCACTGTACTGTTGACAGAAACATAGGACCAATGAATACATACCTGGAGAGCTCCAGTGAGAGTAAAGCAAAATGCTTTTTAAAGATTTCATAGGCCAGACGCGGTGGCTCACGCCTGTAATCCCACCACTTCAGGAGGCCAAGGCAGGTGGATCACTTGAGGTCAGGAGTTTGAGACCAGCTTGGCCAACATGGTGAAACCCTGTCTCTACTAAAAATACAAAAGTTAGCCAGGCATGGTAGCACACACCTGTAGTTCCAGCTACTCGGGAGGCTGAGGCACGGAATCGCTTGAACCCAGGAGGTGGAGGTTTCAGTGAGCCAAG

In [ ]:
from google.colab import files
files.download('HPV.fa')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
!wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_19/GRCh37.p13.genome.fa.gz


--2025-07-18 00:59:23--  https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_19/GRCh37.p13.genome.fa.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 804605548 (767M) [application/x-gzip]
Saving to: ‘GRCh37.p13.genome.fa.gz’

GRCh37.p13.genome.f 100%[===================>] 767.33M  1.23MB/s    in 7m 55s  

2025-07-18 01:07:18 (1.62 MB/s) - ‘GRCh37.p13.genome.fa.gz’ saved [804605548/804605548]



In [21]:
# Step 2: Unzip
!gunzip GRCh37.p13.genome.fa.gz

gzip: GRCh37.p13.genome.fa already exists; do you wish to overwrite (y or n)? y
y


In [22]:


# Step 3: Index the FASTA
!samtools faidx GRCh37.p13.genome.fa



In [23]:
# Step 4: Extract sequences using bedtools
!bedtools getfasta -fi GRCh38.primary_assembly.genome.fa -bed VIS_pos_final.bed -fo HTLV_VIS_pos_final.fa


Feature (chr21:48018514-48019514) beyond the length of chr21 size (46709983 bp).  Skipping.
Feature (chr21:47210542-47211542) beyond the length of chr21 size (46709983 bp).  Skipping.
Feature (chr21:46746402-46747402) beyond the length of chr21 size (46709983 bp).  Skipping.
Feature (chr21:47875445-47876445) beyond the length of chr21 size (46709983 bp).  Skipping.
Feature (chr8:145153728-145154728) beyond the length of chr8 size (145138636 bp).  Skipping.
Feature (chr8:145709628-145710628) beyond the length of chr8 size (145138636 bp).  Skipping.
Feature (chr4:190523952-190524952) beyond the length of chr4 size (190214555 bp).  Skipping.
Feature (chr19:59034397-59035397) beyond the length of chr19 size (58617616 bp).  Skipping.
Feature (chr9:140169741-140170741) beyond the length of chr9 size (138394717 bp).  Skipping.
Feature (chr19:59085120-59086120) beyond the length of chr19 size (58617616 bp).  Skipping.
Feature (chr9:140186029-140187029) beyond the length of chr9 size (138394717

In [24]:
# prompt: count entries in HTLV_VIS_pos_final.fa

with open("HTLV_VIS_pos_final.fa", "r") as f:
    count = sum(1 for line in f if line.startswith('>'))
print(f"Number of entries: {count}")

Number of entries: 31681


In [26]:
# prompt: VIS_neg_final.bed = only n rows

!head -n 31681 VIS_neg_final.bed > subset_lines.bed


In [27]:
!bedtools getfasta -fi GRCh38.primary_assembly.genome.fa -bed subset_lines.bed -fo HTLV_VIS_neg_final.fa

WARNING. chromosome (chr6_qbl_hap6) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr6_dbb_hap3) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr6_cox_hap2) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr9_gl000199_random) was not found in the FASTA file. Skipping.
Feature (chr22:51103998-51104998) beyond the length of chr22 size (50818468 bp).  Skipping.
WARNING. chromosome (chr1_gl000191_random) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr6_cox_hap2) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr17_gl000203_random) was not found in the FASTA file. Skipping.
WARNING. chromosome (chr6_cox_hap2) was not found in the FASTA file. Skipping.
WARNING. chromosome (chrUn_gl000220) was not found in the FASTA file. Skipping.
Feature (chr22:51003107-51004107) beyond the length of chr22 size (50818468 bp).  Skipping.
Feature (chr21:48077833-48078833) beyond the length of chr21 size (46709983 bp).  

In [33]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
import random
from google.colab import files

def label_fasta(input_file, label_value):
    """Reads a FASTA file and returns a list of labeled SeqRecord objects with label=<0|1> in description."""
    labeled_records = []
    for record in SeqIO.parse(input_file, "fasta"):
        # Create a new ID and description that includes label=
        new_id = f"{record.id}"
        new_description = f"{record.description} label={label_value}"
        labeled_record = SeqRecord(
            seq=record.seq,
            id=new_id,
            description=new_description
        )
        labeled_records.append(labeled_record)
    return labeled_records

# Label the positive (label=1) and negative (label=0) sequences
pos_records = label_fasta("HTLV_VIS_pos_final.fa", 1)
neg_records = label_fasta("HTLV_VIS_neg_final.fa", 0)

# Combine and shuffle
combined_records = pos_records + neg_records
random.shuffle(combined_records)

# Save output FASTA
output_filename = "HTLV_combined_shuffled.fa"
SeqIO.write(combined_records, output_filename, "fasta")
print(f"Combined, labeled, and shuffled FASTA saved to {output_filename}")

# Download if on Colab
files.download(output_filename)


Combined, labeled, and shuffled FASTA saved to HTLV_combined_shuffled.fa


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>